## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
import os
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None

In [ ]:
# Google Drive mounting is intentionally not used in this repository.

In [ ]:
!pip install accelerate -U
!pip install transformers[torch]
!pip install bitsandbytes
!pip install -qU peft datasets

In [ ]:
import torch
import random
import time
import pandas as pd
import numpy as np
import nltk
import string
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback, BitsAndBytesConfig
from transformers import AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import bitsandbytes as bnb

In [ ]:
# Κατέβασμα των απαραίτητων δεδομένων από το nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')  # Για σωστό lemmatization
nltk.download('averaged_perceptron_tagger')  # POS tagging
nltk.download('punkt')  # Tokenizer
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# Εισαγωγή του Access Token απο την Hugging face.
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Function to clean text
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def clean_text(text):
    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    words = word_tokenize(text)
    filtered_words = [word for word in words if word not in stop_words]
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in filtered_words]
    return " ".join(lemmatized_words)

In [ ]:
# Load dataset
df = pd.read_csv("data/external/sentences_dataset_45269.csv")


In [ ]:
"""print("\nΠροτάσεις πριν τον καθαρισμό!\n")
print(df.head(30))

df["sentence"] = df["sentence"].apply(clean_text)

print("\nΠροτάσεις μετά τον καθαρισμό!\n")
print(df.head(30))"""

In [ ]:
# Αφαίρεση διπλότυπων εγγραφών. Διατηρείται μόνο η πρώτη εμφάνιση της εν λόγω γραμμής.

# Υπολογισμός αριθμού διπλών εγγραφών.
num_duplicates = df['sentence'].duplicated().sum()
print("Πλήθος διπλών εγγραφών στη στήλη sentence:", num_duplicates)

# Αποθήκευση του αρχικού πλήθους γραμμών.
before = len(df)

# Αφαίρεση διπλότυπων (κρατάμε την πρώτη εμφάνιση).
df = df.drop_duplicates(subset='sentence', keep='first')

# Υπολογισμός του πόσες γραμμές διαγράφηκαν.
after = len(df)
deleted = before - after
print("Πλήθος εγγραφών που διαγράφηκαν:", deleted)

In [ ]:
# Split dataset
train_data, temp_data = train_test_split(df, test_size=0.3, stratify=df['polarity'], random_state=SEED)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=temp_data['polarity'], random_state=SEED)

# Εμφάνιση των διαστάσεων των σετ δεδομένων.
print("Διαστάσεις Σετ Εκπαίδευσης:", train_data.shape)
print("Διαστάσεις Σετ Ελέγχου:", test_data.shape)

In [ ]:
# LLaMA 3 με 8 δισεκατομμύρια παραμέτρους (8B).
MODEL_NAME = "meta-llama/Llama-3.1-8B"

EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))

# Δεκτές εποχές απο 3 έως 20.
while not EPOCHS >= 3 or not EPOCHS <=20:
    EPOCHS = int(input("Δώσε αριθμό εποχών εκπαίδευσης "))


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Use 4-bit quantization
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, fix_mistral_regex=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    #num_labels=3,
    quantization_config=bnb_config,
    output_hidden_states=True,
    #trust_remote_code=True,
    device_map="auto"
)

# Ορισμός του pad_token_id στο configuration του μοντέλου
base_model.config.pad_token_id = tokenizer.pad_token_id

# Freeze LM head so it doesn't receive gradients
for param in base_model.lm_head.parameters():
    param.requires_grad = False

In [ ]:
# LoRA fine-tuning setup
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

base_model = get_peft_model(base_model, lora_config)

# Αριθμός των εκπαιδεύσιμων παραμέτρων λόγω Lora.
base_model.print_trainable_parameters()

In [ ]:
# Προσθήκη κεφαλής ταξινόμησης
import torch.nn as nn

class LLaMaForClassification(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size

        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fct = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden_states = outputs.hidden_states[-1]  # [B, T, H]

        # --- MEAN POOLING ---
        # Προσοχή: το attention_mask πρέπει να είναι float για το πολλαπλασιασμό
        if attention_mask is None:
            pooled = hidden_states.mean(dim=1)
        else:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)

        logits = self.classifier(pooled)

        # Loss
        loss = None
        if labels is not None:
            labels = labels.to(torch.long)
            loss = self.loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

model = LLaMaForClassification(base_model, num_labels=3)

In [ ]:
# Tokenization
def tokenize_function(examples):
    return tokenizer(examples, padding="max_length", truncation=True, max_length=256)

train_encodings = tokenize_function(train_data['sentence'].tolist())
test_encodings = tokenize_function(test_data['sentence'].tolist())
val_encodings = tokenize_function(val_data['sentence'].tolist())

In [ ]:
# Create dataset class
class PolarityDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = PolarityDataset(train_encodings, train_data['polarity'].tolist())
test_dataset = PolarityDataset(test_encodings, test_data['polarity'].tolist())
val_dataset = PolarityDataset(val_encodings, val_data['polarity'].tolist())

In [ ]:
from transformers import EarlyStoppingCallback

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    run_name="Sentiment_Analysis_LLaMA_3",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    #warmup_steps=300,
    warmup_ratio=0.05,
    weight_decay=0.02,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_safetensors=False, # <-- avoids the llama error
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_accumulation_steps=8,
    report_to="none", # να μην γράφει σε wandb
    save_total_limit=2,
    max_grad_norm=1.0,
    learning_rate=2e-5,
    fp16=True,
    optim="adamw_torch_fused" # γρήγορος και σταθερός optimizer
)

#my_optimizer = bnb.optim.Adam32bit(model.parameters(), lr=2e-5) # 2e-5

# Custom loss logger class
import math

class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        # θα κρατήσουμε όλα τα logs με epoch, loss, eval_loss
        self.records = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        epoch = logs.get("epoch", None)
        loss = logs.get("loss", None)
        eval_loss = logs.get("eval_loss", None)

        # κρατάμε μόνο logs που έχουν epoch ΚΑΙ τουλάχιστον ένα από loss / eval_loss
        if epoch is not None and (loss is not None or eval_loss is not None):
            self.records.append({
                "epoch": epoch,
                "loss": loss,
                "eval_loss": eval_loss
            })

loss_logger = LossLoggerCallback()

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    #tokenizer=tokenizer,
    #optimizers = (my_optimizer, None),
    callbacks=[loss_logger, EarlyStoppingCallback(early_stopping_patience=4)]
)

# Ορισμός του χρόνου έναρξης
start_time = time.time()

# Train model
# Εκπαίδευση του μοντέλου
print("Ξεκινά η εκπαίδευση του μοντέλου...\n")

torch.cuda.empty_cache()

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

trainer.train()

# Ορισμός του χρόνου λήξης
end_time = time.time()

# Υπολογισμός του συνολικού χρόνου εκπαίδευσης
total_time = end_time - start_time

# Μετατροπή σε λεπτά & δευτερόλεπτα.
# Χρησιμοποιούμε // 60 για να βρούμε τα λεπτά και % 60 για τα υπόλοιπα δευτερόλεπτα.
# Το % είναι ο τελεστής υπολοίπου MOD.
# Το // κάνει ακέραια διαίρεση, επιστρέφοντας μόνο το ακέραιο μέρος.
minutes = int(total_time // 60)
seconds = int(total_time % 60)
print(f"\nΣυνολικός Χρόνος Εκπαίδευσης: {minutes} λεπτά και {seconds} δευτερόλεπτα ({training_args.num_train_epochs} εποχές)")

### Οπτικοποίηση του Training & Validation Loss ###
"""plt.figure(figsize=(8, 5))
plt.plot(loss_logger.epochs, loss_logger.train_losses, label="Training Loss", marker='o')
plt.plot(loss_logger.epochs, loss_logger.val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.grid(True)
plt.show()"""


# Τα raw logs από το callback
records = loss_logger.records

df_logs = pd.DataFrame(records)

# Διαχωρίζουμε train και val
train_df = df_logs[df_logs["loss"].notna()][["epoch", "loss"]]
train_df = train_df.rename(columns={"loss": "train_loss"})

eval_df = df_logs[df_logs["eval_loss"].notna()][["epoch", "eval_loss"]]
eval_df = eval_df.rename(columns={"eval_loss": "val_loss"})

# Κάνουμε merge ανά epoch
df_loss = pd.merge(train_df, eval_df, on="epoch", how="left")

# Τακτοποιούμε / στρογγυλοποιούμε
df_loss = df_loss.sort_values("epoch").reset_index(drop=True)
df_loss["train_loss"] = df_loss["train_loss"].astype(float)
df_loss["val_loss"] = df_loss["val_loss"].astype(float)

print("To CSV είναι: ")
print(df_loss)

NAME = "LLaMA_3"
# Αποθήκευση σε CSV για αυτό το μοντέλο
csv_path = f"data/processed/logs/loss_{NAME}.csv"
df_loss.to_csv(csv_path, index=False)
print("Saved loss CSV to:", csv_path)


In [ ]:
# Αποθήκευση του εκπαιδευμένου μοντέλου LLaMA 3.
torch.save(model.state_dict(), 'external_materials/model_weights/LLaMA_3/saved_model/saved_llama3_state_dict.pth')
tokenizer.save_pretrained('external_materials/model_weights/LLaMA_3/saved_tokenizer', legacy_format=False)

# Evaluate on test set
predictions = trainer.predict(test_dataset)
predicted_classes = np.argmax(predictions.predictions, axis=-1)

# Display classification report
report = classification_report(test_data['polarity'], predicted_classes, target_names=['Negative', 'Neutral', 'Positive'])
print(report)

with open(f"outputs/results/classification_reports/{NAME}_classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

# Ενοποιημένο μέγεθος γραμματοσειράς
plt.rcParams.update({'font.size': 11})
plt.figure(figsize=(6, 4))

# Display confusion matrix
cm = confusion_matrix(test_data['polarity'], predicted_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
#plt.title('Confusion Matrix - llama')

plt.savefig(
    f'outputs/figures/fig/{NAME}_heatmap_SA_citation.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()

In [ ]:
###############################